# Heart Disease Prediction

Predict heart disease risk using **k-Nearest Neighbors (kNN)** with model tuning and evaluation.

## Problem Statement

Classify whether a patient has heart disease based on clinical measurements and symptoms.

## Dataset Information

This dataset includes patient attributes such as age, chest pain type, blood pressure, cholesterol, and ECG results.

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix

sns.set_theme(style="whitegrid")

In [ ]:
heart_df = pd.read_csv("heart.csv")
heart_df.head()

In [ ]:
print("Dataset shape:", heart_df.shape)
print("\nMissing values:\n", heart_df.isnull().sum())
heart_df.info()
heart_df.describe()

## Exploratory Data Analysis (EDA)

Review target balance and key feature relationships.

In [ ]:
plt.figure(figsize=(8, 4))
sns.countplot(data=heart_df, x="target")
plt.title("Heart Disease Target Distribution")
plt.xlabel("Target")
plt.ylabel("Count")
plt.xticks([0, 1], ["No Disease", "Disease"])
plt.show()

plt.figure(figsize=(10, 4))
sns.histplot(data=heart_df, x="age", bins=20, kde=False)
plt.title("Age Distribution")
plt.xlabel("Age")
plt.ylabel("Count")
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))
sns.countplot(data=heart_df, x="cp", hue="target")
plt.title("Chest Pain Type by Heart Disease")
plt.xlabel("Chest Pain Type")
plt.ylabel("Count")
plt.legend(title="Target")
plt.show()

plt.figure(figsize=(8, 5))
sns.boxplot(data=heart_df, x="target", y="chol")
plt.title("Cholesterol by Heart Disease Status")
plt.xlabel("Target")
plt.ylabel("Cholesterol")
plt.show()

In [ ]:
plt.figure(figsize=(12, 10))
sns.heatmap(heart_df.corr(), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Matrix")
plt.show()

## Feature Selection and Preprocessing

Separate features from the target and prepare the data for modeling.

In [ ]:
X = heart_df.drop("target", axis=1)
y = heart_df["target"]

print("Feature shape:", X.shape)
print("Target distribution:\n", y.value_counts(normalize=True))

## Train-Test Split

Hold out 20% of the data for final evaluation while preserving class proportions.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

## Model Building and Hyperparameter Tuning

Create a pipeline with scaling and kNN, then tune the number of neighbors and weights.

In [ ]:
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier())
])

param_grid = {
    "knn__n_neighbors": [3, 5, 7, 9],
    "knn__weights": ["uniform", "distance"]
}

grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

grid_search.fit(X_train, y_train)
print("Best Parameters:", grid_search.best_params_)
print("Best CV Accuracy: {:.2f}%".format(grid_search.best_score_ * 100))

## Model Evaluation

Evaluate the selected kNN model on the test set and examine classification metrics.

In [ ]:
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)

print("Accuracy: {:.2f}%".format(accuracy_score(y_test, y_pred) * 100))
print("Precision: {:.2f}%".format(precision_score(y_test, y_pred) * 100))
print("Recall: {:.2f}%".format(recall_score(y_test, y_pred) * 100))
print("F1 Score: {:.2f}%".format(f1_score(y_test, y_pred) * 100))

print("\nClassification Report:\n", classification_report(y_test, y_pred))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 4))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["No Disease", "Disease"],
    yticklabels=["No Disease", "Disease"]
)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()

## Results

The tuned kNN classifier provides a reliable baseline for heart disease screening with balanced evaluation metrics and a clear confusion matrix.